# Verification Notebook
This notebook contains the verification tests performed for our MILP problem.

They follow from Sargeant 1998 criteria for verification and validation of simulation models.

Three types of tests are proposed:
1. Unit tests on the deterministic pre-processing.
2. Extreme/Degenerate condition tests.
3. Infeasibility tests.

In [1]:
import numpy as np
import copy
import gurobipy as gp
from gurobipy import GRB

## 1. Unit Tests (Deterministic Pre-Processing)

### phi() - Rocket Equation

In [2]:
# Dummy Network
Connections = {0: [0,1], 1: [0,1]}      # 2 nodes
g_0 = 9.80665                           # m/s^2

delta_V = {0: {0: 0, 1: 2.0},                # ΔV [km/s]
           1: {0: 2.0, 1: 0}}

# Dummy Vehicle Set
I_sp    = np.array([0, 300])

In [3]:
# Phi Function Definition - As in MILP code
def phi(i, j, v, dV=delta_V, I_sp=I_sp, g_0=g_0):
    if I_sp[v] == 0:
        return 1
    else:
        return 1 - np.exp(-(1000 * dV[i][j] / (I_sp[v] * g_0)))

In [4]:
# TEST 1: Zero-Isp Special Case

test_1 = phi(0, 1, 0);              # node 0 to node 1; vehicle 0 (Isp = 0s)
expected_1 = 1

assert test_1 == expected_1, f"TEST 1 - FAIL: expected {expected_1}, got {test_1}"
print(f"TEST 1 - PASS: phi(Isp=0) = {test_1} (expected {expected_1})")

TEST 1 - PASS: phi(Isp=0) = 1 (expected 1)


In [5]:
# TEST 2: Manual Rederivation of the Rocket Equation

i, j, v = 0, 1, 1                   # node 0 to node 1; vehicle 1 (Isp = 300s)
dV_ms = delta_V[i][j] * 1000        # km/s to m/s conversion
expected_2 = 1 - np.exp(-dV_ms / (I_sp[v] * g_0))

test_2 = phi(i, j, v)

assert test_2 == expected_2, f"TEST 2 - FAIL: expected {expected_2}, got {test_2}"
print(f"TEST 2 - PASS: phi = {test_2:.4f} (expected {expected_2:.4f})")

TEST 2 - PASS: phi = 0.4933 (expected 0.4933)


In [6]:
# TEST 3: Holdover Arc
# Staying at the same node should never burn propellant, for any vehicle

result_v0 = phi(0, 0, 0)            # Node 0 to Node 0; Vehicle 0 (Isp = 0s)
result_v1 = phi(0, 0, 1)            # Node 0 to Node 0, Vehicle 1 (Isp = 300s)

assert result_v0 == 1, f"TEST 3 - FAIL: expected 1 for vehicle 0 (Isp = 0s), got {result_v0}"
assert result_v1 == 0, f"TEST 3 - FAIL: expected 0 for vehicle 1 (Isp = 300s) and zero dV, got {result_v1}"

print(f"TEST 3 - PASS: phi(holdover, Isp=0s) = {result_v0}, phi(holdover, Isp=300s) = {result_v1}")

TEST 3 - PASS: phi(holdover, Isp=0s) = 1, phi(holdover, Isp=300s) = 0.0


### AllPossibleOutflowArcs()

In [7]:
# AllpossibleOutflowArcs function from MILP code

def AllpossibleOutflowArcs(Connections, T_adv, window, TOFused):
  

  AllArcs = {}

  for t in T_adv:
    TimeNode = {}
    for i in Connections:
      if t in window[i]:
        
        Now = window[i].index(t)
        TimeNode[i] = {}
        
        for j in Connections[i]:
            
            if t+TOFused[i][j] in window[j]:
              TimeNode[i][j]= {"ArrivalTime": t+TOFused[i][j], "FullTravelTime":TOFused[i][j] }

        #If the holding arc is not available, then the arc to the next available
        #free time block is added, as long as we are not at the end of the window (N_Window[i])
        if (i not in TimeNode[i]) and (Now+1 != len(window[i])): 
          TimeNode[i][i]={"ArrivalTime":window[i][Now+1],"FullTravelTime":window[i][Now+1] - t}
        
        
    if TimeNode != {}:
      AllArcs[t] = TimeNode
            
                    
    
  return AllArcs

In [8]:
# TEST 4: Full Hand-Traced Network

Connection_4    = {0: [0, 1], 1: [0, 1]}
N_Window_4      = {0: [0, 2], 1: [0, 3]}
TOF_4           = {0: {0: 1, 1: 2}, 1: {0: 2, 1: 1}}
T_adv_4         = [0, 1, 2, 3]

result_4 = AllpossibleOutflowArcs(Connection_4, T_adv_4, window=N_Window_4, TOFused=TOF_4)

expected_4 = {
    0: {
        0: {0: {"ArrivalTime": 2, "FullTravelTime": 2}},
        1: {0: {"ArrivalTime": 2, "FullTravelTime": 2},
            1: {"ArrivalTime": 3, "FullTravelTime": 3}},
    },
    2: {0: {}},
    3: {1: {}},
}

assert result_4 == expected_4, f"TEST 4 - FAIL: \ngot {result_4}\nexpected {expected_4}"
print("TEST 4 - PASS: full hand-traced arc set matches exactly")
print(result_4)

TEST 4 - PASS: full hand-traced arc set matches exactly
{0: {0: {0: {'ArrivalTime': 2, 'FullTravelTime': 2}}, 1: {0: {'ArrivalTime': 2, 'FullTravelTime': 2}, 1: {'ArrivalTime': 3, 'FullTravelTime': 3}}}, 2: {0: {}}, 3: {1: {}}}


In [9]:
# TEST 5: Arrival Outside Destination Window

Connections_5   = {0: [1], 1: [0]}
N_Window_5      = {0: [0], 1: [5]}
TOF_5           = {0: {1: 2}, 1: {0: 2}}
T_adv_5         = list(range(6))

result_5 = AllpossibleOutflowArcs(Connections_5, T_adv_5, window=N_Window_5, TOFused=TOF_5)

assert 0 in result_5, "TEST 5 - FAIL: expected an entry for t=0"
assert result_5[0][0] == {}, f"TEST 5 -FAIL: expected no reachable arcs from node 0 at t=0, got {result_5[0][0]}"
assert 1 not in result_5[0][0], "TEST 5 -FAIL: arc to node 1 should not exist -- arrival time misses its window"

print("TEST 5 - PASS: arc correctly absent when arrival time falls outside destination window.")
print(result_5)

TEST 5 - PASS: arc correctly absent when arrival time falls outside destination window.
{0: {0: {}}, 5: {1: {}}}


In [10]:
# TEST 6: Holdover Fallback, Unequal Gap Sizes

Connections_6   = {0: [0]}
N_Window_6      = {0: [0, 4, 9]}
TOF_6           = {0: {0: 100}}
T_adv_6         = list(range(10))

result_6 = AllpossibleOutflowArcs(Connections_6, T_adv_6, window=N_Window_6, TOFused=TOF_6)

expected_6 = {
    0: {0: {0: {"ArrivalTime": 4, "FullTravelTime": 4}}},
    4: {0: {0: {"ArrivalTime": 9, "FullTravelTime": 5}}},
    9: {0: {}},
}

assert result_6 == expected_6, f"TEST 6 - FAIL:\ngot {result_6}\nexpected {expected_6}"

print(f"TEST 6 - PASS: Holdover fallback arcs correctly sized for unequal window gaps, and correctly withheld at the final window entry")
print(result_6)

TEST 6 - PASS: Holdover fallback arcs correctly sized for unequal window gaps, and correctly withheld at the final window entry
{0: {0: {0: {'ArrivalTime': 4, 'FullTravelTime': 4}}}, 4: {0: {0: {'ArrivalTime': 9, 'FullTravelTime': 5}}}, 9: {0: {}}}


### Solo_SC_Consumption_NodV()

In [11]:
# Dummy Commodity Set
crew_mass       = 100
consumption     = 2.0       # kg/crew/day
PropIndex       = 4
CommodityMassConversion = [crew_mass, 1, 1, 1, 1]
Carriable = {0: "INTEGER", 1: "INTEGER"}

# Addition Dummy Network
TOF = {0: {0: 1, 1: 1},
       1: {0: 1, 1: 1}}

# Addition Dummy Vehicle Set
StructureMass = np.array([1000, 2000])

In [12]:
# Solo_SC_Consumption_NodV function from MILP code

def Solo_SC_Consumption_NodV(i, j, consumption=consumption, TOF=TOF, extraPayload=Carriable):

    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [0, 0, 0, 0, 1, 0], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock

    for i1,(x1) in enumerate(extraPayload.items()):
        i2 = len(extraPayload) - i1
        FullMatrix[-i2,-i2] = 1


    return FullMatrix

In [13]:
# TEST 7: Matrix Structure, Normal Usage

i, j = 0, 0
matrix_nodv = Solo_SC_Consumption_NodV(i, j)

expected_nodv = np.zeros((8, 8))
CommodityBlock_expected = np.array([
    [1, 0, 0, 0, 0, 0],                                 # crew
    [-consumption * TOF[i][j], 1, 0, 0, 0, 0],          # consumables
    [0, 0, 1, 0, 0, 0],                                 # equipment
    [0, 0, 0, 1, 0, 0],                                 # samples
    [0, 0, 0, 0, 1, 0],                                 # propellant (unchanged -- no burn) 
    [0, 0, 0, 0, 0, 1],                                 # spacecraft count
])
expected_nodv[:6, :6]   = CommodityBlock_expected
expected_nodv[6,6]      = 1
expected_nodv[7,7]      = 1

assert np.allclose(matrix_nodv, expected_nodv), f"TEST 7 - FAIL:\ngot\n{matrix_nodv}\nexpected:\n{expected_nodv}"
print("TEST 7 - PASS: Holdover consumption matrix matches Eq.(8) with phi=0 (no burn)")

TEST 7 - PASS: Holdover consumption matrix matches Eq.(8) with phi=0 (no burn)


### Solo_SC_Consumption()

In [14]:
# Solo_SC_Consumption function from MILP code

def Solo_SC_Consumption(i, j,v, consumption = consumption, TOF = TOF,structure_mass = StructureMass, extraPayload = Carriable,PropellantIndex = PropIndex):


    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft

    
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [crew_mass * -1*phi(i,j,v,delta_V,I_sp,g_0),
                       -1*phi(i,j,v,delta_V,I_sp,g_0),
                         -1*phi(i,j,v,delta_V,I_sp,g_0),
                           -1*phi(i,j,v,delta_V,I_sp,g_0),
                             1-1*phi(i,j,v,delta_V,I_sp,g_0),
                                -1*structure_mass[v]*phi(i,j,v,delta_V,I_sp,g_0)], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock




    
    for i1,(x1,vtype) in enumerate(extraPayload.items()):
        i2 = len(extraPayload) - i1
        FullMatrix[-i2,-i2] = 1
        FullMatrix[PropellantIndex, -i2] = -1*structure_mass[x1]*phi(i,j,v,delta_V,I_sp,g_0)


    return FullMatrix


In [15]:
# TEST 8: Propellant Row Matches Hand-Derived Rocket Equation

i, j, v = 0, 1, 1
dV_ms = delta_V[i][j] * 1000
phi_expected = 1 - np.exp(-dV_ms / (I_sp[v] * g_0))

matrix_burn = Solo_SC_Consumption(i, j, v)

expected_row = np.array([
    -crew_mass * phi_expected,
    -phi_expected,
    -phi_expected,
    -phi_expected,
    1 - phi_expected,
    -StructureMass[v] * phi_expected,
    -StructureMass[0] * phi_expected,
    -StructureMass[1] * phi_expected,
])

assert np.allclose(matrix_burn[4,:], expected_row), f"TEST 8 - FAIL:\ngot: {matrix_burn[4,:]}\nexpected: {expected_row}"
print(f"TEST 8 - PASS: Propellant row matches hand-derived rocket equation (phi={phi_expected:.4f}).")

TEST 8 - PASS: Propellant row matches hand-derived rocket equation (phi=0.4933).


In [16]:
# TEST 9: Isp=0 special case

i, j, v = 0, 1, 0
matrix_isp0 = Solo_SC_Consumption(i, j, v)

assert matrix_isp0[4,4] == 0, f"TEST 9 - FAIL: Propellant self-retention term = {matrix_isp0[4,4]}, expected 0."
assert matrix_isp0[4,5] == -StructureMass[v], f"TEST 9 - FAIL: Own structure mass term = {matrix_isp0[4,5]}, expected {-StructureMass[v]}."

print("TEST 9 - PASS: Isp=0 vehicle forces phi=1. Propellant row's self-retention term is exactly 0.")

TEST 9 - PASS: Isp=0 vehicle forces phi=1. Propellant row's self-retention term is exactly 0.


### create_concurrency_constraint()

In [17]:
# create_concurrency_constraint function from MILP code

def create_concurrency_constraint(connect = Connections, PropellantCommodityIndex = PropIndex,
                                   MassConversion = CommodityMassConversion,
                                     SCstructMass =StructureMass, payloadSC = Carriable): # Same for all vehicles, max payload mass
    
    #There are 3 separate capacities to keep in mind: Payload, Propellant, and Payload + Prop so 3 separate rows are made 1 for each
    Payloadrow = copy.deepcopy(MassConversion) 
    Payloadrow[PropellantCommodityIndex] = 0 #Massconversion is used for all classic commodities, then the propellant is removed

    Proprow = [0] * len(MassConversion)
    Proprow[PropellantCommodityIndex] = 1

    Combinedrow = copy.deepcopy(MassConversion)

    for i1,v1 in enumerate(payloadSC):
        Payloadrow.append(SCstructMass[v1])
        Proprow.append(0)
        Combinedrow.append(SCstructMass[v1])

    H = [{j: np.array([Payloadrow, #payload
                        Proprow,
                        Combinedrow]) #Propellant
           for j in connect[i]}
          for i in connect]
    return H

In [18]:
# TEST 10: Hand-derived rows

H = create_concurrency_constraint()

expected_payload_row        = np.array([100, 1, 1, 1, 0, 1000, 2000])
expected_prop_row           = np.array([0, 0, 0, 0, 1, 0, 0])
expected_combined_row       = np.array([100, 1, 1, 1, 1, 1000, 2000])

assert len(H) ==2, f"TEST 10 - FAIL: Expected 2 entried (one per origin node), got {len(H)}"
for i in [0, 1]:
    for j in [0, 1]:
        M = H[i][j]
        assert M.shape == (3, 7), f"TEST 10 - FAIL: H[{i}][{j}] shape {M.shape}, expected (3,7)"
        assert np.array_equal(M[0], expected_payload_row),   f"FAIL: payload row at H[{i}][{j}]"
        assert np.array_equal(M[1], expected_prop_row),      f"FAIL: propellant row at H[{i}][{j}]"
        assert np.array_equal(M[2], expected_combined_row),  f"FAIL: combined row at H[{i}][{j}]"

print("TEST 10 - PASS: Concurrency matrix matches hand-derived rows, identical across every (i,j) arc.")
print(H[0][0])

TEST 10 - PASS: Concurrency matrix matches hand-derived rows, identical across every (i,j) arc.
[[ 100    1    1    1    0 1000 2000]
 [   0    0    0    0    1    0    0]
 [ 100    1    1    1    1 1000 2000]]


### create_sc_design_parameters()

In [19]:
# create_sc_design_parameters function from MILP code

def create_sc_design_parameters(V, PayloadCap, PropCapacity,payloadSC =Carriable):
    e = np.zeros((V,3,1+len(payloadSC)))

    for i1, e1 in enumerate(e): #First 3 
            e[i1][0][0] = PayloadCap[i1]
            e[i1][1][0] = PropCapacity[i1]
            e[i1][2][0] = PayloadCap[i1] +PropCapacity[i1]


            for i2,c1 in enumerate(payloadSC): #additional values for the payload variables
                 e[i1][1][1+i2] = PropCapacity[c1] 

    #e = [np.array([[PayloadCap[v]],
    #                [PropCapacity[v]],
    #                [PayloadCap[v]+PropCapacity[v]]]) for v in range(V)]
    
    

    return e

In [20]:
# Additional vehicle definitions
PayloadCap      = np.array([500, 800])
PropCapacity    = np.array([0, 5000])

In [21]:
# TEST 11: Hand-derived e array

e = create_sc_design_parameters(V=2, PayloadCap=PayloadCap, PropCapacity=PropCapacity)

expected_e = np.array([
    [[500, 0, 0],       # vehicle 0: payload cap, [own prop cap, carried-0 prop, carried-1 prop]
     [0, 0, 5000],
     [500, 0, 0]],
    [[800, 0, 0],       # vehicle 1
     [5000, 0, 5000],
     [5800, 0, 0]],
])

assert e.shape == (2, 3, 3), f"TEST 11 - FAIL: Shape {e.shape}, expected (2,3,3)"
assert np.array_equal(e, expected_e), f"TEST 11 - FAIL:\ngot\n{e}\nexpected:\n{expected_e}"

print("TEST 11 - PASS: Design-parameter array matches hand-derived payload/propellant/combined capacities.")
print(e)

TEST 11 - PASS: Design-parameter array matches hand-derived payload/propellant/combined capacities.
[[[ 500.    0.    0.]
  [   0.    0. 5000.]
  [ 500.    0.    0.]]

 [[ 800.    0.    0.]
  [5000.    0. 5000.]
  [5800.    0.    0.]]]


## 2. Extreme/Degenerate Conditions Test

### Passthrough Invariants Hold Even After Extreme Burn

In [22]:
# TEST 12

extreme_delta_V = 25.0
delta_V = {0: {0: 0, 1: extreme_delta_V}, 1: {0: extreme_delta_V, 1: 0}}

i, j, v = 0, 1, 1

phi_expected = 1 - np.exp(-(1000 * delta_V[i][j]) / (I_sp[v] * g_0))
print(f"Phi at extreme delta-V: {phi_expected:.6f}")

m2 = gp.Model("Test_2_1_extreme_burn_passthrough")
m2.Params.OutputFlag = 0

# Outflow: what's loaded onto the vehicle at node i (fixed values)
crew_out        = m2.addVar(vtype=GRB.INTEGER,    lb=3,    ub=3,    name="crew_out")
consumables_out = m2.addVar(vtype=GRB.CONTINUOUS, lb=50,   ub=50,   name="consumables_out")
equipment_out   = m2.addVar(vtype=GRB.CONTINUOUS, lb=20,   ub=20,   name="equipment_out")
samples_out     = m2.addVar(vtype=GRB.CONTINUOUS, lb=15,   ub=15,   name="samples_out")
propellant_out  = m2.addVar(vtype=GRB.CONTINUOUS, lb=4500, ub=4500, name="propellant_out")
y_out           = m2.addVar(vtype=GRB.INTEGER,    lb=1,    ub=1,    name="y_out")
scpay_out0      = m2.addVar(vtype=GRB.INTEGER,    lb=0,    ub=0,    name="scpayload_out_0")
scpay_out1      = m2.addVar(vtype=GRB.INTEGER,    lb=0,    ub=0,    name="scpayload_out_1")

# Inflow: What arrives at node j -- determined entirely by the constraint
crew_in        = m2.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="crew_in")
consumables_in = m2.addVar(vtype=GRB.CONTINUOUS, lb=0,      ub=10000, name="consumables_in")
equipment_in   = m2.addVar(vtype=GRB.CONTINUOUS, lb=0,      ub=10000, name="equipment_in")
samples_in     = m2.addVar(vtype=GRB.CONTINUOUS, lb=0,      ub=10000, name="samples_in")
propellant_in  = m2.addVar(vtype=GRB.CONTINUOUS, lb=-10000, ub=10000, name="propellant_in")
y_in           = m2.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="y_in")
scpay_in0      = m2.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="scpayload_in_0")
scpay_in1      = m2.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="scpayload_in_1")

m2.update()

Vout = np.array([crew_out, consumables_out, equipment_out, samples_out, propellant_out,
                  y_out, scpay_out0, scpay_out1])
Vin  = np.array([crew_in, consumables_in, equipment_in, samples_in, propellant_in,
                  y_in, scpay_in0, scpay_in1])

Consumed = Solo_SC_Consumption(i, j, v)
transformed = np.dot(Consumed, Vout)

for k in range(len(transformed)):
    m2.addConstr(transformed[k] == Vin[k], name=f"transform_{k}")

m2.setObjective(0, GRB.MINIMIZE)
m2.optimize()
assert m2.Status == GRB.OPTIMAL, f"Expected OPTIMAL, got status {m2.Status}"

print(f"crew_in={crew_in.X}, equipment_in={equipment_in.X}, "
      f"samples_in={samples_in.X}, y_in={y_in.X}")
print(f"propellant_in={propellant_in.X:.1f}")

assert crew_in.X == 3,       f"TEST 12 - FAIL: crew changed under extreme burn, got {crew_in.X}"
assert equipment_in.X == 20, f"TEST 12 - FAIL: equipment changed under extreme burn, got {equipment_in.X}"
assert samples_in.X == 15,   f"TEST 12 - FAIL: samples changed under extreme burn, got {samples_in.X}"
assert y_in.X == 1,          f"TEST 12 - FAIL: spacecraft count changed under extreme burn, got {y_in.X}"

print("\nTEST 12 - PASS: crew, equipment, samples, and spacecraft count all pass through unchanged.")
print("          Final fuel may be negative due to extreme dV definition")

Phi at extreme delta-V: 0.999796
Set parameter Username
Set parameter LicenseID to value 2739971
Academic license - for non-commercial use only - expires 2026-11-17
crew_in=3.0, equipment_in=20.0, samples_in=15.0, y_in=1.0
propellant_in=-2383.6

TEST 12 - PASS: crew, equipment, samples, and spacecraft count all pass through unchanged.
          Final fuel may be negative due to extreme dV definition


### Consumables Depletion is Decoupled from Propellant Burn

In [23]:
# TEST 13

crew_val = 3
consumables_start = 50
expected_loss = consumption * TOF[0][0] * crew_val

print(f"Expected consumables loss (crew={crew_val}, TOF={TOF[0][0]}d): {expected_loss} kg")
print(f"Consumables at the start: {consumables_start} kg")
print(f"Expected consumables_in on both arcs: {consumables_start - expected_loss} kg")

def build_and_solve(name, i, j, v, use_burn_matrix):
    mm = gp.Model(name)
    mm.Params.OutputFlag = 0

    crew_out        = mm.addVar(vtype=GRB.INTEGER,    lb=crew_val, ub=crew_val, name="crew_out")
    consumables_out = mm.addVar(vtype=GRB.CONTINUOUS, lb=consumables_start, ub=consumables_start, name="consumables_out")
    equipment_out   = mm.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=0, name="equipment_out")
    samples_out     = mm.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=0, name="samples_out")
    propellant_out  = mm.addVar(vtype=GRB.CONTINUOUS, lb=4500, ub=4500, name="propellant_out")
    y_out           = mm.addVar(vtype=GRB.INTEGER,    lb=1, ub=1, name="y_out")
    scpay_out0      = mm.addVar(vtype=GRB.INTEGER,    lb=0, ub=0, name="scpayload_out_0")
    scpay_out1      = mm.addVar(vtype=GRB.INTEGER,    lb=0, ub=0, name="scpayload_out_1")

    crew_in        = mm.addVar(vtype=GRB.INTEGER,    lb=0, ub=1000, name="crew_in")
    consumables_in = mm.addVar(vtype=GRB.CONTINUOUS, lb=-1000, ub=1000, name="consumables_in")
    equipment_in   = mm.addVar(vtype=GRB.CONTINUOUS, lb=-1000, ub=1000, name="equipment_in")
    samples_in     = mm.addVar(vtype=GRB.CONTINUOUS, lb=-1000, ub=1000, name="samples_in")
    propellant_in  = mm.addVar(vtype=GRB.CONTINUOUS, lb=-100000, ub=100000, name="propellant_in")
    y_in           = mm.addVar(vtype=GRB.INTEGER,    lb=0, ub=1000, name="y_in")
    scpay_in0      = mm.addVar(vtype=GRB.INTEGER,    lb=0, ub=1000, name="scpayload_in_0")
    scpay_in1      = mm.addVar(vtype=GRB.INTEGER,    lb=0, ub=1000, name="scpayload_in_1")

    mm.update()

    Vout = np.array([crew_out, consumables_out, equipment_out, samples_out, propellant_out,
                      y_out, scpay_out0, scpay_out1])
    Vin  = np.array([crew_in, consumables_in, equipment_in, samples_in, propellant_in,
                      y_in, scpay_in0, scpay_in1])

    Consumed = Solo_SC_Consumption(i, j, v) if use_burn_matrix else Solo_SC_Consumption_NodV(i, j)

    transformed = np.dot(Consumed, Vout)
    for k in range(len(transformed)):
        mm.addConstr(transformed[k] == Vin[k], name=f"transform_{k}")

    mm.setObjective(0, GRB.MINIMIZE)
    mm.optimize()
    assert mm.Status == GRB.OPTIMAL, f"{name}: expected OPTIMAL, got {mm.Status}"

    return consumables_in.X

consumables_in_holdover = build_and_solve("Test_2_4b_holdover", 0, 0, 1, use_burn_matrix=False)
consumables_in_burn     = build_and_solve("Test_2_4b_burn",     0, 1, 1, use_burn_matrix=True)

print(f" -> consumables_in (holdover, no burn):     {consumables_in_holdover} kg")
print(f" -> consumables_in (0->1 arc, ~99.98% burn): {consumables_in_burn} kg")

assert consumables_in_holdover == consumables_start - expected_loss, \
    f"TEST 13 - FAIL: holdover consumables_in={consumables_in_holdover}, expected {consumables_start - expected_loss}"
assert consumables_in_burn == consumables_start - expected_loss, \
    f"TEST 13 - FAIL: burn-arc consumables_in={consumables_in_burn}, expected {consumables_start - expected_loss}"
assert consumables_in_holdover == consumables_in_burn, \
    "TEST 13 - FAIL: consumables loss differs between the no-burn and burn arcs"

print("TEST 13 - PASS: consumables consumption is independent of fuel burn")

Expected consumables loss (crew=3, TOF=1d): 6.0 kg
Consumables at the start: 50 kg
Expected consumables_in on both arcs: 44.0 kg
 -> consumables_in (holdover, no burn):     44.0 kg
 -> consumables_in (0->1 arc, ~99.98% burn): 44.0 kg
TEST 13 - PASS: consumables consumption is independent of fuel burn


### Carried Spacecraft's Structure Mass Taxes the Carrier's Propellant

In [24]:
# TEST 14

delta_V = {0: {0: 0, 1: 2.0}, 1: {0: 2.0, 1: 0}}

i, j, v = 0, 1, 1       # carrier is vehicle 1

phi_expected = 1 - np.exp(-(1000 * delta_V[i][j] / (I_sp[v] * g_0)))
print(f"Carrier's phi (correct):        {phi_expected:.6f}")

crew_val            = 2
consumables_val     = 30
equipment_val       = 0
samples_val         = 0
propellant_val      = 3500

y_val, scpay0_val, scpay1_val = 1, 1, 0

expected_propellant_in = (
    -phi_expected * (crew_mass * crew_val + consumables_val + equipment_val + samples_val)
    + (1 - phi_expected) * propellant_val
    - phi_expected * StructureMass[1] * y_val         # carrier's own structure mass
    - phi_expected * StructureMass[0] * scpay0_val     # carried type-0's structure mass
    - phi_expected * StructureMass[1] * scpay1_val     # carried type-1's structure mass (0 here)
)

wrong_phi = 1.0
wrong_propellant_in = (
    -wrong_phi * (crew_mass * crew_val + consumables_val + equipment_val + samples_val)
    + (1 - wrong_phi) * propellant_val
    - wrong_phi * StructureMass[1] * y_val
    - wrong_phi * StructureMass[0] * scpay0_val
    - wrong_phi * StructureMass[1] * scpay1_val
)
print(f"Expected propellant_in (correct phi):                            {expected_propellant_in:.2f} kg")
print(f"Propellant_in if carried Isp used instead of carrier (bug):     {wrong_propellant_in:.2f} kg")

Carrier's phi (correct):        0.493287
Expected propellant_in (correct phi):                            180.18 kg
Propellant_in if carried Isp used instead of carrier (bug):     -3230.00 kg


In [25]:
m3 = gp.Model("Test_2_4c_carried_spacecraft")
m3.Params.OutputFlag = 0

crew_out        = m3.addVar(vtype=GRB.INTEGER,    lb=crew_val,        ub=crew_val,        name="crew_out")
consumables_out = m3.addVar(vtype=GRB.CONTINUOUS, lb=consumables_val, ub=consumables_val, name="consumables_out")
equipment_out   = m3.addVar(vtype=GRB.CONTINUOUS, lb=equipment_val,   ub=equipment_val,   name="equipment_out")
samples_out     = m3.addVar(vtype=GRB.CONTINUOUS, lb=samples_val,     ub=samples_val,     name="samples_out")
propellant_out  = m3.addVar(vtype=GRB.CONTINUOUS, lb=propellant_val,  ub=propellant_val,  name="propellant_out")
y_out           = m3.addVar(vtype=GRB.INTEGER,    lb=y_val,           ub=y_val,           name="y_out")
scpay_out0      = m3.addVar(vtype=GRB.INTEGER,    lb=scpay0_val,      ub=scpay0_val,      name="scpayload_out_0")
scpay_out1      = m3.addVar(vtype=GRB.INTEGER,    lb=scpay1_val,      ub=scpay1_val,      name="scpayload_out_1")

crew_in        = m3.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="crew_in")
consumables_in = m3.addVar(vtype=GRB.CONTINUOUS, lb=-1000,  ub=1000,  name="consumables_in")
equipment_in   = m3.addVar(vtype=GRB.CONTINUOUS, lb=-1000,  ub=1000,  name="equipment_in")
samples_in     = m3.addVar(vtype=GRB.CONTINUOUS, lb=-1000,  ub=1000,  name="samples_in")
propellant_in  = m3.addVar(vtype=GRB.CONTINUOUS, lb=-10000, ub=10000, name="propellant_in")
y_in           = m3.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="y_in")
scpay_in0      = m3.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="scpayload_in_0")
scpay_in1      = m3.addVar(vtype=GRB.INTEGER,    lb=0,      ub=1000,  name="scpayload_in_1")

m3.update()

Vout = np.array([crew_out, consumables_out, equipment_out, samples_out, propellant_out,
                  y_out, scpay_out0, scpay_out1])
Vin  = np.array([crew_in, consumables_in, equipment_in, samples_in, propellant_in,
                  y_in, scpay_in0, scpay_in1])

Consumed = Solo_SC_Consumption(i, j, v)
transformed = np.dot(Consumed, Vout)

for k in range(len(transformed)):
    m3.addConstr(transformed[k] == Vin[k], name=f"transform_{k}")

m3.setObjective(0, GRB.MINIMIZE)
m3.optimize()
assert m3.Status == GRB.OPTIMAL, f"Expected OPTIMAL, got status {m3.Status}"

print(f"Solved propellant_in:                   {propellant_in.X:.2f} kg")
print(f"Expected propellant_in:                 {expected_propellant_in:.2f} kg")

assert np.isclose(propellant_in.X, expected_propellant_in), \
    f"TEST 14 - FAIL: propellant_in={propellant_in.X:.2f}, expected {expected_propellant_in:.2f}"

print("TEST 14 - PASS: Carried spacecraft's structure mass is charged against the carrier's propellant.")

Solved propellant_in:                   180.18 kg
Expected propellant_in:                 180.18 kg
TEST 14 - PASS: Carried spacecraft's structure mass is charged against the carrier's propellant.
